## Download the reference assets


Run these commands in this directory:

```bash
PREFIX="https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle"
curl -fL -o hair_classifier_v1.onnx.data "${PREFIX}/hair_classifier_v1.onnx.data"
curl -fL -o hair_classifier_v1.onnx "${PREFIX}/hair_classifier_v1.onnx"
curl -fL -o sample.jpeg \
  "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
```

Verify the files against `asset_manifest.json`:

```bash
sha256sum hair_classifier_v1.onnx hair_classifier_v1.onnx.data sample.jpeg
```

The ONNX file and its external-data file must remain next to each other with
these exact names. The sample image is 1024x1024 RGB JPEG. The manifest records
all three checksums and the model graph interface.

Install the pinned local dependencies if you want to run the handler directly:

```bash
python -m pip install -r requirements.txt
```

## Question 1: look at the ONNX graph, output name


Load the model with ONNX Runtime and look at `session.get_inputs()` and
`session.get_outputs()`. What is the output node name?

- `output`
- `sigmoid`
- `softmax`
- `prediction`

The input node is `input` and has shape `(batch, 3, 200, 200)`. The output is a
single float: the probability of the `straight` class (`straight=1`).

```
python -c "
import onnxruntime as ort
s = ort.InferenceSession('hair_classifier_v1.onnx', providers=['CPUExecutionProvider'])
print('inputs :', [(i.name, i.shape, i.type) for i in s.get_inputs()])
print('outputs:', [(o.name, o.shape, o.type) for o in s.get_outputs()])
"
```

The output is:
```
inputs : [('input', ['s77', 3, 200, 200], 'tensor(float)')]
outputs: [('output', ['s77', 1], 'tensor(float)')]
```

The answer is: output

## Question 2: target size

What target size does `prepare_image` use?

- `64x64`
- `128x128`
- `200x200`
- `256x256`

The answer is: 200x200

## Image preprocessing

Use the same preprocessing as the fixed deep-learning reference setup:

In [ ]:
from io import BytesIO
from urllib import request

import numpy as np
import onnxruntime as ort
from PIL import Image

SAMPLE_URL = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"


def download_image(url):
    with request.urlopen(url, timeout=30) as response:
        return Image.open(BytesIO(response.read())).convert("RGB")


def prepare_image(image):
    image = image.resize((200, 200), Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array(
        [0.229, 0.224, 0.225], dtype=np.float32
    )
    return np.transpose(array, (2, 0, 1))[None, ...]


session = ort.InferenceSession(
    "hair_classifier_v1.onnx", providers=["CPUExecutionProvider"]
)
print("inputs :", [(i.name, i.shape) for i in session.get_inputs()])
print("outputs:", [(o.name, o.shape) for o in session.get_outputs()])  

tensor = prepare_image(download_image(SAMPLE_URL))
print("Q3 tensor[0,0,0,0] =", round(float(tensor[0, 0, 0, 0]), 3))     

output = session.run(["output"], {"input": tensor})[0]
print("Q4 probability     =", round(float(output.ravel()[0]), 3))       

inputs : [('input', ['s77', 3, 200, 200])]
outputs: [('output', ['s77', 1])]
Q3 tensor[0,0,0,0] = -1.056
Q4 probability     = 0.728


The interpolation method, RGB conversion, channel order, normalization, and
`float32` conversion are part of the interface.

In 2025 the interpolation was left implicit in one homework and set to `NEAREST` in the next one, which made the first pixel and model output disagree. 

Here both modules explicitly use bilinear resizing.

## Question 3: first normalized R value

Download `sample.jpeg`, run `prepare_image`, and report the first value of the R channel (`tensor[0, 0, 0, 0]`) rounded to three decimal places. 

The grader accepts an absolute error of `0.01`.

The answer is: -1.056

## Question 4: local ONNX inference probability

Report the output probability rounded to three decimal places. 

The grader accepts an absolute error of `0.02`. Because the model, image, and preprocessing are checksummed and CPU inference is used, this is no longer a choice between platform-dependent options.

The answer is: 0.728

## Lambda handler

The provided `lambda_function.py` implements the complete handler. Read it and test it locally with:

```bash
python smoke_test.py
```

The handler accepts either a direct event or an API Gateway-style event. The request payload is:

```json
{
  "image_url": "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
}
```

It returns HTTP status `200` and a JSON body containing `straight_probability` and the boolean `straight` prediction.

## Container

Build the Lambda-compatible image from the checked-in Dockerfile:

```bash
docker build -t mlzoomcamp-2026-serverless .
docker run --rm -p 9000:8080 mlzoomcamp-2026-serverless
```

Invoke it from another terminal:

```bash
curl -s \
  -XPOST 'http://localhost:9000/2015-03-31/functions/function/invocations' \
  -H 'Content-Type: application/json' \
  -d '{"image_url":"https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"}'
```

The Dockerfile uses the AWS Lambda Python 3.13 base image, installs the
exact runtime dependencies from `requirements-lambda.txt`, and copies the same
checksummed ONNX pair. It does not substitute an empty or different model.

## Question 5: look at the Lambda configuration base image

Which runtime base image is declared in the Dockerfile?

- `public.ecr.aws/lambda/python:3.9`
- `public.ecr.aws/lambda/python:3.11`
- `public.ecr.aws/lambda/python:3.13`
- `python:3.13-slim-bookworm`

The local image's displayed size is intentionally not graded: it depends on
architecture, Docker version, and cached layers.

The answer is: public.ecr.aws/lambda/python:3.13

## Question 6: invoke the container inference probability

Inspect the JSON string in the Lambda response body and report
`straight_probability` rounded to three decimal places. 

The grader accepts an
absolute error of `0.02`. It must match Question 4 because both paths use the
same model, image, preprocessing, and CPU execution provider.

```
{"statusCode": 200, "headers": {"Content-Type": "application/json"}, "body": "{\"straight_probability\": 0.727697, \"straight\": true}"}(ZoomML_Homeworks)
```

The answer is: 0.728